# 

# Data preperation

author: @Embra-Schuilenburg

COMP 432: Main Project

This document will process the data from the "Separable Neurocomputational Mechanisms Underlying Multisensory Learning" dataset saved in the relative foler "original_dataset".

This notebook will:
1. Preproccess the dataset using fMRIPrep
2. Use Nilearn to further preprocess dat & convert it into ROI time series
3. Convert fMRI runs into feature vectors 
4. Define a dataset object which will store the datapoints and labels used in future project ML experiments 
5. Save the dataset object that can be loaded later without needing to re-run this notebook

## Imports and packages

In [1]:
from helper_functions import load_saved_run
import os
import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, List, Dict, Any
import glob
from __future__ import annotations
import numpy as np
import pandas as pd
import nibabel as nib
import nilearn as nl
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.interfaces.fmriprep import load_confounds_strategy
from nilearn.maskers import NiftiLabelsMasker

---

## 1. Preprocessing

Preprocessing will be done with fMRIPrep using the following flags:
- Hardware flags:
    - NTHREADS=8
    - OMP_NTHREADS=4
    - MEM_MB=24000
- Processing flags:
    - --fs-no-reconall 
    - --skip-bids-validation
    - --output-spaces MNI152NLin2009cAsym 

In [2]:
# Set vars
flag = True  # DO NOT SET TO FALSE UNLESS RE-RUNNING PREPROCESSING, PROCESSING TIME > 12HRS
# flag = False
os.chdir('/media/peripheral/Code/Comp432/machine-learning-mechanism-of-multisensory-learning/')

In [3]:
# Run script
if not flag:
    os.chdir('/media/peripheral/Code/Comp432/machine-learning-mechanism-of-multisensory-learning/scripts')
    os.system('./run-fmriprep.sh')
    os.chdir(
        '/media/peripheral/Code/Comp432/machine-learning-mechanism-of-multisensory-learning/')
    flag = True

---

# 2. Nilearn processing and ROI extraction

A pipeline will be used to extract features for future machine learning applications from the outputs of the previous steps organized as follows:

- raw BIDS dataset: `original-dataset/`
- fMRIPrep derivatives: `/media/peripheral/Code/Comp432/dataset/derivatives/`

Pipeline parameters:

- BOLD space: `MNI152NLin2009cAsym`
- Atlas: Schaefer-100
- Confounds: Nilearn fMRIPrep `simple`
- Temporal filtering: detrend + high-pass only
- Event representation: short event window per event for CNN input

Outputs:

- one `.npz` per run containing ROI time series and event windows
- one `.tsv` per run for labels
- one dataset manifest

## Pipeline

Resources:
- https://nilearn.github.io/stable/manipulating_images/input_output.html
- https://nilearn.github.io/dev/auto_examples/03_connectivity/plot_signal_extraction.html
- https://nilearn.github.io/dev/manipulating_images/masker_objects.html
- https://nilearn.github.io/dev/modules/generated/nilearn.datasets.fetch_atlas_schaefer_2018.html
- https://nilearn.github.io/dev/modules/generated/nilearn.maskers.NiftiLabelsMasker.html
- https://nilearn.github.io/stable/modules/generated/nilearn.interfaces.fmriprep.load_confounds.html
- https://nilearn.github.io/dev/connectivity/functional_connectomes.html#confounds-regression

In [4]:
# Project paths

PROJECT_DIR = Path(
    '/media/peripheral/Code/Comp432/machine-learning-mechanism-of-multisensory-learning/')
ORIG_DATASET_DIR = PROJECT_DIR / 'original-dataset'
DERIVATIVES_DIR = Path('/media/peripheral/Code/Comp432/dataset/derivatives/')
OUTPUT_DIR = PROJECT_DIR / 'processed_data'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load atlas

def load_schaefer_100_atlas():
    atlas = fetch_atlas_schaefer_2018(
        n_rois=100,
        yeo_networks=7,
        resolution_mm=2
    )

    atlas_maps = atlas.maps
    atlas_labels = [
        lab.decode('utf-8') if isinstance(lab, bytes) else str(lab)
        for lab in atlas.labels
    ]

    # Remove background label if present
    if len(atlas_labels) > 0 and atlas_labels[0].lower() == 'background':
        atlas_labels = atlas_labels[1:]

    return atlas_maps, atlas_labels


atlas_maps, atlas_labels = load_schaefer_100_atlas()
print(f'Number of ROIs: {len(atlas_labels)}')
print(atlas_labels[:5])

[fetch_atlas_schaefer_2018] Dataset found in /home/emxn/nilearn_data/schaefer_2018

Number of ROIs: 100
['7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4', '7Networks_LH_Vis_5']


In [6]:
# File path helpers

def get_subject_ids(original_dataset_dir):
    """
    Return sorted subject IDs like ['sub-01', 'sub-02', ...]
    based on the original dataset directory.
    """
    return sorted([p.name for p in original_dataset_dir.iterdir() if p.is_dir() and p.name.startswith('sub-')])


def get_run_file_paths(subject_id, run_number):
    """
    Build the expected paths for one subject/run.

    Returns
    -------
    dict with keys:
        bold, bold_json, confounds_tsv, events_tsv
    """
    run_str = f'run-{run_number}'

    bold = DERIVATIVES_DIR / subject_id / 'func' / (
        f'{subject_id}_task-learn_{run_str}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz'
    )
    bold_json = DERIVATIVES_DIR / subject_id / 'func' / (
        f'{subject_id}_task-learn_{run_str}_space-MNI152NLin2009cAsym_desc-preproc_bold.json'
    )
    confounds_tsv = DERIVATIVES_DIR / subject_id / 'func' / (
        f'{subject_id}_task-learn_{run_str}_desc-confounds_timeseries.tsv'
    )
    events_tsv = ORIG_DATASET_DIR / subject_id / 'func' / (
        f'{subject_id}_task-learn_{run_str}_events.tsv'
    )

    return {
        'bold': bold,
        'bold_json': bold_json,
        'confounds_tsv': confounds_tsv,
        'events_tsv': events_tsv,
    }


def run_files_exist(run_paths):
    """Check whether all required files exist for a run."""
    return all(path.exists() for path in run_paths.values())


def get_available_runs(subject_id, max_runs=10):
    """
    Return all run numbers that exist for a subject.
    """
    available = []
    for run_number in range(1, max_runs + 1):
        run_paths = get_run_file_paths(subject_id, run_number)
        if run_files_exist(run_paths):
            available.append(run_number)
    return available

In [7]:
# Read repetition time (TR)

def load_tr_from_bold_json(bold_json_path):
    """
    Load TR from the fMRIPrep preprocessed BOLD sidecar JSON.
    """
    with open(bold_json_path, 'r') as f:
        meta = json.load(f)

    tr = meta.get('RepetitionTime', None)
    if tr is None:
        raise ValueError(f'RepetitionTime not found in {bold_json_path}')

    return float(tr)

In [8]:
# Load simple fMRIPrep confounds

def load_simple_confounds(confounds_tsv_path, use_global_signal=True):
    """
    Load a simple confound set from fMRIPrep confounds TSV.

    Included by default:
    - 6 rigid-body motion params
    - white_matter
    - csf
    - optional global_signal
    - any non_steady_state_outlier columns

    Returns
    -------
    confounds_df : pd.DataFrame
        Numeric confounds matrix ready for nuisance regression.
    """
    confounds = pd.read_csv(confounds_tsv_path, sep='\t')

    base_cols = [
        'trans_x', 'trans_y', 'trans_z',
        'rot_x', 'rot_y', 'rot_z',
        'white_matter', 'csf'
    ]

    if use_global_signal and 'global_signal' in confounds.columns:
        base_cols.append('global_signal')

    nonsteady_cols = [c for c in confounds.columns if c.startswith(
        'non_steady_state_outlier')]

    selected_cols = [
        c for c in base_cols if c in confounds.columns] + nonsteady_cols

    if len(selected_cols) == 0:
        raise ValueError(
            f'No expected confound columns found in {confounds_tsv_path}')

    confounds_df = confounds[selected_cols].copy()

    # Fill NaNs conservatively so masker regression does not fail
    confounds_df = confounds_df.fillna(0.0)

    return confounds_df

---

# 2. Feature construction

Using the ROI time series from Nilearn this report will now:
- Align time series to run events
- Build feature vectors for each run
- Normalize features within runs (performed by the masker)

In [ ]:
# Extract ROI time series for one run

def extract_run_roi_timeseries(
    bold_path,
    atlas_maps,
    confounds_df=None,
    standardize='zscore_sample',  # Specified to use in nilearn documentation
    detrend=True
):
    """
    Extract ROI time series from one preprocessed BOLD run. During extraction signals are normalized by the masker, which: 

    1. centers each time series 
    2. scales each time series (within a run)
    3. reduces scale across ROIs and runs

    This run-level standardization ensures that ROI activations within a run have approximately means = 0 and unit variance across time. 

    Parameters
    ----------
    bold_path : path-like
        Preprocessed fMRIPrep BOLD NIfTI in MNI space.
    atlas_maps : path-like
        Schaefer atlas label image.
    confounds_df : pd.DataFrame or None
        Confounds to regress out.
    standardize : bool
        Standardize each ROI time series.
    detrend : bool
        Detrend ROI time series.

    Returns
    -------
    roi_ts : np.ndarray, shape (n_timepoints, n_rois)
    """
    masker = NiftiLabelsMasker(
        labels_img=atlas_maps,
        standardize=standardize,
        detrend=detrend,
        t_r=None,   # not applying explicit temporal filtering here, performed at event alignment
        verbose=0
    )

    roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)
    return roi_ts

In [10]:
# Load and prepare events

def load_run_events(events_tsv_path):
    """
    Load the run events TSV and keep relevant columns.

    Required columns:
    - onset
    - duration
    - trial_type
    - param_surprise
    - param_V

    Returns
    -------
    events_df : pd.DataFrame
    """
    events = pd.read_csv(events_tsv_path, sep='\t')

    required_cols = ['onset', 'duration',
                     'trial_type', 'param_surprise', 'param_V']
    missing = [c for c in required_cols if c not in events.columns]
    if missing:
        raise ValueError(f'Missing columns in {events_tsv_path}: {missing}')

    # Keep only trials with both regression targets present
    events = events.dropna(
        subset=['onset', 'param_surprise', 'param_V']).copy()
    events = events.reset_index(drop=True)

    return events


def add_event_timing_columns(events_df, tr):
    """
    Keep timing in seconds and add helpful TR-boundary columns.

    Note: Changed from previous extraction method to not collapse onset to a single
    rounded TR. Original method can be viewed in back up in archive folder.
    """
    events = events_df.copy()

    events['onset_sec'] = events['onset'].astype(float)
    events['duration_sec'] = events['duration'].astype(float)
    events['offset_sec'] = events['onset_sec'] + events['duration_sec']

    # Optional inspection columns
    events['onset_tr_floor'] = np.floor(events['onset_sec'] / tr).astype(int)
    events['offset_tr_ceil'] = np.ceil(events['offset_sec'] / tr).astype(int)

    return events

In [11]:
# Build trial feature vectors from run ROI time series

def extract_trial_features_and_labels(
    roi_ts,
    events_df,
    tr,
    hrf_lag_trs=2,
    pre_event_sec=0.0,
    post_event_sec=0.0
):
    """
    Convert run-level ROI time series into one RT-aware averaged feature vector per trial.

    Note: changed from previous version to create an averaged ROI vector from a choice 
    duration dependant interval rather than a fixed repetition time window to model windows
    more faithfully to the original publication.

    For each choice event:
        raw choice epoch = [onset, onset + duration]
        BOLD-aligned epoch = [onset + hrf_lag, onset + duration + hrf_lag]

    Optional padding:
        start_sec = onset + hrf_lag - pre_event_sec
        end_sec   = onset + duration + hrf_lag + post_event_sec

    We keep timing in seconds until the final conversion to TR indices.
    That avoids losing sub-TR timing too early.

    Parameters
    ----------
    roi_ts : np.ndarray, shape (n_timepoints, n_rois)
    events_df : pd.DataFrame
        Must include onset_sec and duration_sec (or onset/duration), plus labels.
    tr : float
        Repetition time in seconds.
    hrf_lag_trs : int
        HRF lag measured in TRs. Converted internally to seconds.
    pre_event_sec : float
        Optional seconds to include before the lagged choice epoch.
    post_event_sec : float
        Optional seconds to include after the lagged choice epoch.

    Returns
    -------
    X : np.ndarray, shape (n_trials_kept, n_rois)
    y : np.ndarray, shape (n_trials_kept, 2)
    trial_meta : pd.DataFrame
        Trial metadata aligned row-wise with X and y.
    """
    n_tp, n_rois = roi_ts.shape
    hrf_lag_sec = hrf_lag_trs * tr

    X_list = []
    y_list = []
    kept_rows = []
    meta_rows = []

    for idx, row in events_df.iterrows():
        onset_sec = float(
            row['onset_sec']) if 'onset_sec' in row else float(row['onset'])
        duration_sec = float(
            row['duration_sec']) if 'duration_sec' in row else float(row['duration'])

        # RT-aware lagged choice epoch
        start_sec = onset_sec + hrf_lag_sec - pre_event_sec
        end_sec = onset_sec + duration_sec + hrf_lag_sec + post_event_sec

        # Convert to inclusive TR indices only at the end
        start_tr = int(np.floor(start_sec / tr))
        end_tr = int(np.ceil(end_sec / tr) - 1)

        # Ensure at least one TR is included
        if end_tr < start_tr:
            end_tr = start_tr

        # Skip if window falls outside run
        if start_tr < 0 or end_tr >= n_tp:
            continue

        trial_window = roi_ts[start_tr:end_tr + 1, :]   # inclusive
        trial_feature = trial_window.mean(axis=0)       # shape: (n_rois,)

        X_list.append(trial_feature)
        y_list.append([row['param_surprise'], row['param_V']])
        kept_rows.append(idx)

        meta_row = row.copy()
        meta_row['feature_start_sec'] = start_sec
        meta_row['feature_end_sec'] = end_sec
        meta_row['feature_start_tr'] = start_tr
        meta_row['feature_end_tr'] = end_tr
        meta_row['feature_n_trs'] = end_tr - start_tr + 1
        meta_rows.append(meta_row)

    if len(X_list) == 0:
        raise ValueError('No valid trials remained after RT-aware windowing.')

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    trial_meta = pd.DataFrame(meta_rows).reset_index(drop=True)

    return X, y, trial_meta


# Process one run end-to-end

def process_single_run(
    subject_id,
    run_number,
    atlas_maps,
    use_global_signal=True,
    standardize='zscore_sample',
    detrend=True,
    hrf_lag_trs=2,
    pre_event_sec=0.0,
    post_event_sec=0.0
):
    """
    Process one subject/run into trial-level features and labels.

    Note: this method was updated to process features from the choice dependant
    window rather than the fixed length one.

    Returns
    -------
    run_data : dict
        {
            'X': np.ndarray (n_trials, n_rois),
            'y': np.ndarray (n_trials, 2),
            'trial_meta': pd.DataFrame,
            'tr': float,
            'subject_id': str,
            'run_number': int
        }
    """
    run_paths = get_run_file_paths(subject_id, run_number)
    if not run_files_exist(run_paths):
        raise FileNotFoundError(
            f'Missing files for {subject_id}, run-{run_number}')

    tr = load_tr_from_bold_json(run_paths['bold_json'])
    confounds_df = load_simple_confounds(
        run_paths['confounds_tsv'], use_global_signal=use_global_signal)
    roi_ts = extract_run_roi_timeseries(
        bold_path=run_paths['bold'],
        atlas_maps=atlas_maps,
        confounds_df=confounds_df,
        standardize=standardize,
        detrend=detrend
    )

    events_df = load_run_events(run_paths['events_tsv'])
    events_df = add_event_timing_columns(events_df, tr)

    X, y, trial_meta = extract_trial_features_and_labels(
        roi_ts=roi_ts,
        events_df=events_df,
        tr=tr,
        hrf_lag_trs=hrf_lag_trs,
        pre_event_sec=pre_event_sec,
        post_event_sec=post_event_sec
    )

    return {
        'X': X,
        'y': y,
        'trial_meta': trial_meta,
        'tr': tr,
        'subject_id': subject_id,
        'run_number': run_number
    }


# Save one processed run

def save_run_data(run_data, output_dir, atlas_labels):
    """
    Save one run as:
    - features_and_labels.npz  (arrays for training)
    - trial_metadata.csv       (readable trial table)

    Directory structure:
    processed_data/
        sub-01/
            run-1/
                features_and_labels.npz
                trial_metadata.csv
    """
    subject_id = run_data['subject_id']
    run_number = run_data['run_number']

    run_out_dir = output_dir / subject_id / f'run-{run_number}'
    run_out_dir.mkdir(parents=True, exist_ok=True)

    npz_path = run_out_dir / 'features_and_labels.npz'
    csv_path = run_out_dir / 'trial_metadata.csv'

    np.savez_compressed(
        npz_path,
        X=run_data['X'],                     # (n_trials, n_rois)
        y=run_data['y'],                     # (n_trials, 2)
        y_names=np.array(['param_surprise', 'param_V']),
        roi_names=np.array(atlas_labels),
        tr=np.array(run_data['tr']),
        subject_id=np.array(subject_id),
        run_number=np.array(run_number)
    )

    run_data['trial_meta'].to_csv(csv_path, index=False)

    return npz_path, csv_path


# Process all runs for one participant

def process_subject(
    subject_id,
    atlas_maps,
    atlas_labels,
    output_dir,
    max_runs=10,
    use_global_signal=True,
    standardize='zscore_sample',
    detrend=True,
    hrf_lag_trs=2,      # ~4.67 s lag
    pre_event_sec=0.0,
    post_event_sec=0.0
):
    """
    Process every available run for one subject and save outputs, and skip run processing if run already exists.

    Note: this method was updated average BOLD signals over the choice duration only.

    # TODO: consider adding a .5 to 1.0 second padding to the post event portion of the window.
    """
    subject_dir = os.path.join(output_dir, subject_id)
    run_numbers = get_available_runs(subject_id, max_runs=max_runs)
    print(f'{subject_id}: found runs {run_numbers}')

    saved_files = []

    for run_number in run_numbers:
        run_dir = os.path.join(subject_dir, 'run-' + str(run_number))

        # Check if the run was already processed, if so don't do it again
        if (os.path.isdir(run_dir)):
            continue

        try:
            run_data = process_single_run(
                subject_id=subject_id,
                run_number=run_number,
                atlas_maps=atlas_maps,
                use_global_signal=use_global_signal,
                standardize=standardize,
                detrend=detrend,
                hrf_lag_trs=hrf_lag_trs,
                pre_event_sec=pre_event_sec,
                post_event_sec=post_event_sec
            )

            npz_path, csv_path = save_run_data(
                run_data, output_dir, atlas_labels)
            saved_files.append((npz_path, csv_path))

            print(
                f'  saved run-{run_number}: '
                f'X {run_data['X'].shape}, y {run_data['y'].shape}'
            )

        except Exception as e:
            print(f'  skipped run-{run_number}: {e}')

    return saved_files

In [12]:
# Process the whole dataset (output hidden due to length)

subject_ids = get_subject_ids(ORIG_DATASET_DIR)
print(f'Found {len(subject_ids)} subjects')

all_saved = []

for subject_id in subject_ids:
    saved = process_subject(
        subject_id=subject_id,
        atlas_maps=atlas_maps,
        atlas_labels=atlas_labels,
        output_dir=OUTPUT_DIR,
        max_runs=10,
        use_global_signal=True,
        standardize='zscore_sample',
        detrend=True,
        hrf_lag_trs=2,      # ~4.67 s lag
        pre_event_sec=0.0,
        post_event_sec=0.0
    )
    all_saved.extend(saved)

print(f'\nTotal saved run files: {len(all_saved)}')

Found 58 subjects
sub-01: found runs [1, 2, 3, 4, 5, 6]


/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


  saved run-1: X (60, 100), y (60, 2)


/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


  saved run-2: X (60, 100), y (60, 2)


...

/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


  saved run-4: X (60, 100), y (60, 2)


/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


  saved run-5: X (60, 100), y (60, 2)
  saved run-6: X (60, 100), y (60, 2)

Total saved run files: 348


/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


In [13]:
# Inspect one saved run

example_npz = OUTPUT_DIR / 'sub-01' / 'run-1' / 'features_and_labels.npz'
example_csv = OUTPUT_DIR / 'sub-01' / 'run-1' / 'trial_metadata.csv'

data = np.load(example_npz, allow_pickle=True)
trial_meta = pd.read_csv(example_csv)


# Visualize data to ensure it was saved correctly

print('X shape:', data['X'].shape)          # (n_trials, 100)
print('y shape:', data['y'].shape)          # (n_trials, 2)
print('y names:', data['y_names'])
print('First 5 ROI names:', data['roi_names'][:5])

display(trial_meta.head())
display(pd.DataFrame(data['y'], columns=data['y_names']))

X shape: (60, 100)
y shape: (60, 2)
y names: ['param_surprise' 'param_V']
First 5 ROI names: ['7Networks_LH_Vis_1' '7Networks_LH_Vis_2' '7Networks_LH_Vis_3'
 '7Networks_LH_Vis_4' '7Networks_LH_Vis_5']


,onset,duration,trial_type,param_surprise,param_V,param_rpe,onset_sec,duration_sec,offset_sec,onset_tr_floor,offset_tr_ceil,feature_start_sec,feature_end_sec,feature_start_tr,feature_end_tr,feature_n_trs
0,3.477700,2.226177,ChoiceTactile,-0.152112,-0.958429,NaN,3.477700,2.226177,5.703877,1,3,8.145380,10.371557,3,4,2
1,11.100398,1.879317,ChoiceTactile,0.143367,-0.958429,NaN,11.100398,1.879317,12.979715,4,6,15.768078,17.647395,6,7,2
2,19.372291,2.399425,ChoiceTactile,0.410659,-0.958429,NaN,19.372291,2.399425,21.771716,8,10,24.039971,26.439396,10,11,2
3,27.596148,2.111797,ChoiceTactile,0.654678,-0.958429,NaN,27.596148,2.111797,29.707946,11,13,32.263828,34.375626,13,14,2
4,35.176328,1.813504,ChoiceTactile,0.879154,-0.958429,NaN,35.176328,1.813504,36.989832,15,16,39.844008,41.657512,17,17,1


,param_surprise,param_V
0,-0.152112,-0.958429
1,0.143367,-0.958429
2,0.410659,-0.958429
3,0.654678,-0.958429
4,0.879154,-0.958429
5,1.086986,-0.958429
6,1.280473,-0.958429
7,1.461468,-0.958429
8,1.631486,-0.958429
9,-0.152112,-0.965513


In [14]:
# Test
X, y_joint, y_surprise, y_V, roi_names, y_names = load_saved_run(example_npz)

print('Input ROIs shape:', X.shape)
print('Joint target shape:', y_joint.shape)
print('Surprise-only shape:', y_surprise.shape)
print('V-only shape:', y_V.shape)

# Full run matrix printout
full_array = np.concatenate([y_joint, X], axis=1)
full_array.shape
df = pd.DataFrame(full_array, columns=np.concatenate(
    [['Surprise', 'V'], data['roi_names']]))

Input ROIs shape: (60, 100)
Joint target shape: (60, 2)
Surprise-only shape: (60,)
V-only shape: (60,)


---

# 3. Quality Control

Resources:
- https://nilearn.github.io/dev/plotting/index.html
- https://nilearn.github.io/dev/auto_examples/01_plotting/plot_atlas.html


In [15]:
# Intialize testing vars
all_tests_passed = True
tests_failed = np.array([])

In [16]:
# Quality check 1: structural validation of all saved runs

def validate_saved_run(run_dir):
    npz_path = run_dir / 'features_and_labels.npz'
    csv_path = run_dir / 'trial_metadata.csv'

    result = {
        'run_dir': str(run_dir),
        'ok': True,
        'errors': []
    }

    if not npz_path.exists():
        result['ok'] = False
        result['errors'].append('Missing features_and_labels.npz')
        return result

    if not csv_path.exists():
        result['ok'] = False
        result['errors'].append('Missing trial_metadata.csv')
        return result

    try:
        data = np.load(npz_path, allow_pickle=True)
        meta = pd.read_csv(csv_path)

        X = data['X']
        y = data['y']
        y_names = list(data['y_names'])
        roi_names = data['roi_names']

        if X.ndim != 2:
            result['ok'] = False
            result['errors'].append(f'X should be 2D, got shape {X.shape}')

        if y.ndim != 2:
            result['ok'] = False
            result['errors'].append(f'y should be 2D, got shape {y.shape}')

        if X.shape[0] != y.shape[0]:
            result['ok'] = False
            result['errors'].append(
                f'Mismatch: X rows {X.shape[0]} != y rows {y.shape[0]}')

        if X.shape[0] != len(meta):
            result['ok'] = False
            result['errors'].append(
                f'Mismatch: X rows {X.shape[0]} != metadata rows {len(meta)}')

        if X.shape[1] != 100:
            result['ok'] = False
            result['errors'].append(f'Expected 100 ROIs, got {X.shape[1]}')

        if y.shape[1] != 2:
            result['ok'] = False
            result['errors'].append(f'Expected 2 targets, got {y.shape[1]}')

        if y_names != ['param_surprise', 'param_V']:
            result['ok'] = False
            result['errors'].append(f'Unexpected y_names: {y_names}')

        if len(roi_names) != 100:
            result['ok'] = False
            result['errors'].append(
                f'Expected 100 roi_names, got {len(roi_names)}')

        if not np.isfinite(X).all():
            result['ok'] = False
            result['errors'].append('X contains NaN or inf')

        if not np.isfinite(y).all():
            result['ok'] = False
            result['errors'].append('y contains NaN or inf')

    except Exception as e:
        result['ok'] = False
        result['errors'].append(str(e))

    return result


def validate_all_saved_runs(output_dir):
    run_dirs = sorted(
        [p for p in Path(output_dir).glob('sub-*/run-*') if p.is_dir()])
    results = [validate_saved_run(run_dir) for run_dir in run_dirs]
    return pd.DataFrame(results)


validation_df = validate_all_saved_runs(OUTPUT_DIR)

if (not validation_df[~validation_df['ok'] == 'False'].empty):
    # Show successful runs
    display(validation_df)
    # Show failed runs
    print('Failed runs:')
    display(validation_df[~validation_df['ok'] == 'False'])
    all_tests_passed = False
    tests_failed = tests_failed.append(1)
else:
    print('Test 1 passed')

Test 1 passed


In [17]:
# Quality check 2: verify y matches trial_metadata exactly

def check_label_alignment(run_dir):
    npz_path = run_dir / 'features_and_labels.npz'
    csv_path = run_dir / 'trial_metadata.csv'

    data = np.load(npz_path, allow_pickle=True)
    meta = pd.read_csv(csv_path)

    y = data['y']
    meta_y = meta[['param_surprise', 'param_V']].to_numpy(dtype=np.float32)

    same = np.allclose(y, meta_y, equal_nan=False)

    return {
        'run_dir': str(run_dir),
        'labels_match_metadata': same,
        'max_abs_diff': float(np.max(np.abs(y - meta_y))) if len(y) > 0 else 0.0
    }


alignment_results = pd.DataFrame([
    check_label_alignment(run_dir)
    for run_dir in sorted(OUTPUT_DIR.glob('sub-*/run-*'))
])

if (not alignment_results[alignment_results['labels_match_metadata'] == False].empty):
    display(
        alignment_results[alignment_results['labels_match_metadata'] == False])
    all_tests_passed = False
    tests_failed = tests_failed.append(2)
else:
    print('Test 2 passed')

Test 2 passed


In [18]:
# Quality check 3: compare raw event counts to saved trial counts

def summarize_run_trial_counts(subject_id, run_number):
    run_paths = get_run_file_paths(subject_id, run_number)

    all_events = pd.read_csv(run_paths['events_tsv'], sep='\t')
    events = all_events[all_events['param_rpe'] != None]
    total_events = len(events)

    valid_label_events = events.dropna(
        subset=['onset', 'param_surprise', 'param_V'])
    n_valid_label_events = len(valid_label_events)

    saved_npz = OUTPUT_DIR / subject_id / \
        f'run-{run_number}' / 'features_and_labels.npz'
    if saved_npz.exists():
        saved_n_trials = np.load(saved_npz, allow_pickle=True)['X'].shape[0]
    else:
        saved_n_trials = np.nan

    return {
        'subject_id': subject_id,
        'run_number': run_number,
        'total_events': total_events,
        'valid_label_events': n_valid_label_events,
        'saved_trials': saved_n_trials,
        'dropped_after_windowing': n_valid_label_events - saved_n_trials if pd.notna(saved_n_trials) else np.nan
    }


rows = []
for subject_id in get_subject_ids(ORIG_DATASET_DIR):
    for run_number in get_available_runs(subject_id):
        rows.append(summarize_run_trial_counts(subject_id, run_number))

trial_count_df = pd.DataFrame(rows)

if (not trial_count_df[trial_count_df['dropped_after_windowing'] > 0].empty):
    display(trial_count_df[trial_count_df['dropped_after_windowing'] > 0])
    all_tests_passed = False
    tests_failed = tests_failed.append(3)
else:
    print('Test 3 passed')

Test 3 passed


In [19]:
# Summarize results of quality checking
if (not all_tests_passed):
    print('Tests failed: ', tests_failed)
else:
    print('All tests passed!')

All tests passed!


In [20]:
subject_id = 'sub-01'
run_number = 1

run_data = process_single_run(
    subject_id=subject_id,
    run_number=run_number,
    atlas_maps=atlas_maps,
    use_global_signal=True,
    standardize='zscore_sample',
    detrend=True,
    hrf_lag_trs=2,
    pre_event_sec=0.0,
    post_event_sec=0.0
)

print('X shape:', run_data['X'].shape)
print('y shape:', run_data['y'].shape)
print('TR:', run_data['tr'])
run_data['trial_meta'].head()

X shape: (60, 100)
y shape: (60, 2)
TR: 2.3338400000000004


/tmp/ipykernel_254308/2491123346.py:44: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  roi_ts = masker.fit_transform(str(bold_path), confounds=confounds_df)


,onset,duration,trial_type,param_surprise,param_V,param_rpe,onset_sec,duration_sec,offset_sec,onset_tr_floor,offset_tr_ceil,feature_start_sec,feature_end_sec,feature_start_tr,feature_end_tr,feature_n_trs
0,3.477700,2.226177,ChoiceTactile,-0.152112,-0.958429,NaN,3.477700,2.226177,5.703877,1,3,8.145380,10.371557,3,4,2
1,11.100398,1.879317,ChoiceTactile,0.143367,-0.958429,NaN,11.100398,1.879317,12.979715,4,6,15.768078,17.647395,6,7,2
2,19.372291,2.399425,ChoiceTactile,0.410659,-0.958429,NaN,19.372291,2.399425,21.771716,8,10,24.039971,26.439396,10,11,2
3,27.596148,2.111797,ChoiceTactile,0.654678,-0.958429,NaN,27.596148,2.111797,29.707946,11,13,32.263828,34.375626,13,14,2
4,35.176328,1.813504,ChoiceTactile,0.879154,-0.958429,NaN,35.176328,1.813504,36.989832,15,16,39.844008,41.657512,17,17,1


In [21]:
tm = run_data['trial_meta'].copy()

cols = ['onset', 'duration', 'feature_start_tr', 'feature_end_tr', 'feature_n_trs',
        'param_surprise', 'param_V']
print(tm[cols].head(15))

print('\nFeature_n_trs value counts:')
print(tm['feature_n_trs'].value_counts().sort_index())

print('\nDuration summary:')
print(tm['duration'].describe())

print('\nCorrelation between duration and feature_n_trs:')
print(tm[['duration', 'feature_n_trs']].corr())

         onset  duration  feature_start_tr  feature_end_tr  feature_n_trs  \
0     3.477700  2.226177                 3               4              2   
1    11.100398  1.879317                 6               7              2   
2    19.372291  2.399425                10              11              2   
3    27.596148  2.111797                13              14              2   
4    35.176328  1.813504                17              17              1   
5    42.258005  2.308804                20              21              2   
6    51.468402  1.570878                24              24              1   
7    59.467687  2.074120                27              28              2   
8    68.323450  2.040392                31              32              2   
9    77.216086  2.150200                35              36              2   
10   83.810321  2.210418                37              38              2   
11   91.648389  1.934837                41              42              2   

In [22]:
run_paths = get_run_file_paths(subject_id, run_number)
tr = load_tr_from_bold_json(run_paths['bold_json'])

events_df = load_run_events(run_paths['events_tsv'])
events_df = add_event_timing_columns(events_df, tr)

print('Choice events in TSV:', len(events_df))
print('Trials kept after windowing:', len(run_data['trial_meta']))
print('Dropped trials:', len(events_df) - len(run_data['trial_meta']))

Choice events in TSV: 60
Trials kept after windowing: 60
Dropped trials: 0


In [23]:
tm = run_data['trial_meta'].copy()
tm[tm['duration'] <= 0][['onset', 'duration',
                         'trial_type', 'param_surprise', 'param_V']]

,onset,duration,trial_type,param_surprise,param_V
49,407.793275,0.0,ChoiceTactile,-0.758498,0.0
55,456.213164,0.0,ChoiceTactile,-0.107946,0.0


---

# Conclusion

This concludes the data processing notebook. 

In this notebook I: 
1. Ran fMRIPrep preprocessing to denoise runs, correct head motion motion, and prepare data for feature extraction
2. Extracted ROI vectors to use as features for run-level model training by:
    1. Defining helper functions to make the feature extraction process easier, and to facilitate file storage
    2. Extracting run level features which represent model inputs as event-aligned ROI vectors which align trial events to averages of windows of ROI time series with the following specifications:
        - Window includes 0s of pre-event ROIs
        - Window's center is 2s after recorded event time
        - Window captures first 2s from event time to window center, and two seconds after plus an addition 1s to capture any latent activity
3. Tested outputs to ensure processing was successful

Extracted data was saved to the following relative director:
- `processed_data/sub-#/run-#`